# Архитектура Transformer

## Литература
* [Sequence to Sequence (seq2seq) and Attention by Lena Voita](https://lena-voita.github.io/nlp_course/seq2seq_and_attention.html)
* [NLP course by Hugging Face](https://huggingface.co/learn/nlp-course/chapter0/1?fw=pt)
* [Transformer: A Novel Neural Network Architecture for Language Understanding](https://blog.research.google/2017/08/transformer-novel-neural-network.html)
* [Transformers-based Encoder-Decoder Models](https://huggingface.co/blog/encoder-decoder)
* [The Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/)
* [Transformer в картинках](https://habr.com/ru/articles/486358/)

## Архитектура Transformer

Трансформер — архитектура нейронных сетей, представленная в 2017 году исследователями из Google Brain в статье [Attention Is All You Need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). Архитектура трансформера в статье выглядела следующим образом ([изображение взято из блога Лены Войты](https://lena-voita.github.io/nlp_course/seq2seq_and_attention.html)):

<img src="pictures/seq2seq_voita12.png" width=800 height=800 />

В этой архитектуре нет ни сверточных, ни рекуррентных компонентов. По аналогии с Encoder-Decoder архитектурой составляющие Transformer-а можно разложить на следующие части:

* **Encoder**: стопка идентичных слоёв (обычно 6), каждый из которых содержит Multi-Head Attention и Feed-Forward сеть.
* **Decoder**: стопка идентичных слоёв (обычно 6), каждый из которых содержит Masked Multi-Head Attention, Cross-Attention (Multi-Head Attention над выходом энкодера) и Feed-Forward сеть.
* **Связь между encoder и decoder**: Cross-Attention (Multi-Head Attention), где запросы приходят из декодера, а ключи и значения — из энкодера.

Эта архитектура стала SOTA-подходом для задачи машинного перевода, и сейчас де-факто является стандартом во многих задачах, связанных с NLP, CV и т.д. Также эта архитектура широко используется для различных языковых моделей, начало бурного развития которых и положила статья Attention Is All You Need.

> Transformer = attention + pretraining + fine tuning

### Основные архитектурные компоненты

#### 1. Positional Encoding

Self-attention не чувствует порядка слов — если перемешать токены, выход тоже перемешается. Чтобы модель понимала порядок, в Transformer добавляется позиционное кодирование:

- **Что это**: вектор, который добавляется (не конкатенируется!) к эмбеддингу каждого токена.
- **Итоговый вектор**: $h_i = x_i + p_i$, где $x_i$ — эмбеддинг токена, $p_i$ — позиционное кодирование.
- **Требования**: кодирование должно быть одинаковой размерности с эмбеддингом ($d_{model}$), чтобы можно было складывать.

В оригинальной статье используется **синусоидальное кодирование**:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{\text{model}}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$$

**Почему синусоидальное:**
- Для любого фиксированного смещения $k$ позицию $pos+k$ можно выразить как линейную функцию от позиции $pos$, что позволяет модели легче обучаться относительным позициям.
- Не требует обучения (в отличие от learnable positional encoding).

Альтернативный подход — **learnable positional encoding**, где позиционные кодирования — это обучаемые параметры. Этот подход используется в BERT и многих современных моделях.

#### 2. Residual Connections

Каждый подблок в Transformer (Multi-Head Attention и Feed-Forward) имеет skip connection:

$$\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

**Зачем нужно:**
- Позволяет градиентам проходить через сеть напрямую, решая проблему затухающих градиентов.
- Упрощает обучение очень глубоких сетей (до 100+ слоёв).
- Позволяет модели «фокусироваться» на изучении малых изменений, а не на передаче информации целиком.

#### 3. Layer Normalization

После каждого остаточного соединения применяется Layer Normalization:

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sigma} + \beta$$

где $\mu$ и $\sigma$ — среднее и стандартное отклонение по признаковому измерению, $\gamma$ и $\beta$ — обучаемые параметры масштаба и сдвига.

**Зачем LayerNorm, а не BatchNorm:**
- BatchNorm зависит от размера батча и плохо работает с переменной длиной последовательности.
- LayerNorm нормализует каждый объект независимо, что естественно для последовательностей разной длины.

#### 4. Feed-Forward Network

После Multi-Head Attention в каждом слое идёт полносвязная сеть (два линейных слоя с нелинейностью между ними):

$$\text{FFN}(x) = \text{max}(0, xW_1 + b_1)W_2 + b_2$$

В оригинальной статье используется ReLU, но в современных моделях часто применяют GELU, Swish или GLU-варианты.

**Размерность**:
- Вход: $d_{model}$ (обычно 512 или 768)
- Промежуточный слой: $d_{ff}$ = 2048 (в 4 раза больше) — это ключевой гиперпараметр.
- Выход: $d_{model}$

**Зачем FFN:**
- Добавляет нелинейность (attention — линейная операция).
- Расширяет и сжимает представление, позволяя модели изучать сложные паттерны.
- Каждый токен обрабатывается независимо, что добавляет параллелизм.

#### 5. Dropout

В Transformer применяется dropout для регуляризации:
- После каждого подблока (Attention и FFN).
- После добавления позиционного кодирования.
- На весах внимания (attention dropout).

### Полный слой энкодера (Encoder Layer)

1. Вход: последовательность эмбеддингов с позиционным кодированием.
2. **Multi-Head Self-Attention** — каждый токен взаимодействует со всеми токенами последовательности.
3. **Add & Norm** (остаточное соединение + LayerNorm).
4. **Feed-Forward Network** — независимая обработка каждого токена.
5. **Add & Norm** (ещё одно остаточное соединение + LayerNorm).
6. Выход: обновлённые представления токенов.

Повторяется $N$ раз (в оригинале $N=6$).

### Полный слой декодера (Decoder Layer)

1. Вход: последовательность эмбеддингов с позиционным кодированием (сдвинутая вправо для авторегрессии).
2. **Masked Multi-Head Self-Attention** — каждый токен взаимодействует только с предыдущими (causal mask).
3. **Add & Norm**.
4. **Cross-Attention (Encoder-Decoder Attention)** — запросы из декодера, ключи и значения из энкодера.
5. **Add & Norm**.
6. **Feed-Forward Network**.
7. **Add & Norm**.
8. Выход: обновлённые представления для генерации.

Повторяется $N$ раз (в оригинале $N=6$).

### Важные детали реализации

**Размерности:**
- Вход: (batch, seq_len) → эмбеддинги: (batch, seq_len, $d_{model}$)
- Multi-Head Attention: $d_{model}$ делится на $H$ голов, каждая работает с размерностью $d_k = d_{model} / H$
- Выход Multi-Head Attention: (batch, seq_len, $d_{model}$)
- FFN: (batch, seq_len, $d_{model}$) → (batch, seq_len, $d_{ff}$) → (batch, seq_len, $d_{model}$)

**Ключевые гиперпараметры:**
- $d_{model} = 512$ (или 768, 1024, 4096 в больших моделях)
- $H = 8$ (число голов внимания)
- $d_{ff} = 2048$ (размер скрытого слоя в FFN)
- $N = 6$ (число слоёв)
- Dropout rate = 0.1

**Типы внимания в одном Transformer:**
1. **Encoder Self-Attention**: Q, K, V из энкодера (без маски).
2. **Decoder Self-Attention (Masked)**: Q, K, V из декодера (с causal маской).
3. **Cross-Attention**: Q из декодера, K, V из энкодера (без маски, но с возможной padding mask).

### Почему Transformer эффективнее RNN и CNN

Основные архитектурные преимущества Transformer перед RNN и CNN:

1. **Полный параллелизм вычислений** — обработка всех элементов последовательности одновременно (за счёт self-attention), что обеспечивает эффективное масштабирование на GPU/TPU и позволяет обучать модели на больших объёмах данных.

2. **Эффективное решение проблемы затухающего градиента** — остаточные соединения и LayerNorm позволяют обучать очень глубокие модели.

3. **Эффективный Transfer Learning** — минимальное индуктивное смещение (отсутствие привязки к локальности или порядку) делает предобученные представления универсальными; одна модель после донастройки применима к разным языкам, задачам и модальностям с малым количеством размеченных данных.

4. **Единая архитектурная парадигма** — одна и та же конструкция (последовательность + attention) работает с текстом (NLP), изображениями (CV, ViT), аудио и другими сигналами без смены архитектурного стека.

5. **Интерпретируемость через карты внимания** — визуализация весов attention позволяет анализировать, какие элементы входной последовательности влияют на решение (инструмент диагностики, а не строгая каузация).

6. **Эффективный инференс с KV-кэшированием** — при автогрессивной генерации сохранение матриц ключей и значений предыдущих токенов исключает пересчёт всей истории на каждом шаге, снижая вычислительные затраты.

## Языковые модели из Transformer

* [Зоопарк моделей трансформеров на Hugging Face](https://huggingface.co/docs/transformers/model_doc/bert)
* [Transformer models: an introduction and catalog — 2023 Edition](https://amatria.in/blog/transformer-models-an-introduction-and-catalog-2d1e9039f376/)
    * [Catalog table](https://amatria.in/blog/transformer-models-an-introduction-and-catalog-2d1e9039f376/#catalog-table)
    * [Family Tree](https://amatria.in/blog/transformer-models-an-introduction-and-catalog-2d1e9039f376/#family-tree)
    * [Catalog List](https://amatria.in/blog/transformer-models-an-introduction-and-catalog-2d1e9039f376/#catalog-list)
* [A Survey of Transformers](https://arxiv.org/pdf/2106.04554)
* [Pre-trained Models for Natural Language Processing: A Survey](https://arxiv.org/abs/2003.08271)
* [Harnessing the Power of LLMs in Practice: A Survey on ChatGPT and Beyon](https://arxiv.org/pdf/2304.13712)
    * The evolutionary tree of modern LLMs
* [A Survey of Large Language Models](https://arxiv.org/pdf/2303.18223)
* [Formal Algorithms for Transformers](https://arxiv.org/pdf/2207.09238)

Кратко историю развития языковых моделей можно описать так:

| Год | Архитектура | Статья | Исследователи | Примечание |
| --- | --- | --- | --- | --- |
| 2018-06 | **GPT-1** | [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) | OpenAI | 117 миллионов параметров, 5 GB обучающих данных |
| 2018-10 | **BERT** | [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/pdf/1810.04805) | Google AI Language | [Страница BERT на HF](https://huggingface.co/docs/transformers/model_doc/bert) |
| 2019-02 | **GPT-2** | [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) | OpenAI | 1.5 миллиарда параметров, 40 GB обучающих данных |
| 2019-09 | **ALBERT** | [ALBERT: A Lite BERT for Self-supervised Learning of Language Representations](https://arxiv.org/pdf/1909.11942) | 1Google Research | [Страница ALBERT на HF](https://huggingface.co/docs/transformers/model_doc/albert) |
| 2019-10 | **DistilBERT** | [DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter](https://arxiv.org/pdf/1910.01108) | Hugging Face | дистиллированный BERT, на 60% быстрее, на 40% легче по памяти, при этом составляет 97% производительности BERT |
| 2019-10 | **BART** | [BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension](https://arxiv.org/pdf/1910.13461) | Facebook AI | [Страница BART на HF](https://huggingface.co/docs/transformers/model_doc/bart) |
| 2019-10 | **T5** | [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/pdf/1910.10683) | Google AI | [Страница T5 на HF](https://huggingface.co/docs/transformers/model_doc/t5) |
| 2020-05 | **GPT-3** | [Language Models are Few-Shot Learners](https://arxiv.org/pdf/2005.14165.pdf) | OpenAI | 175 миллионов параметров, 45000 GB обучающих данных |

[**Хронологический таймлайн моделей и количества их параметров можно посмотреть здесь.**](https://amatria.in/blog/transformer-models-an-introduction-and-catalog-2d1e9039f376/#chronological-timeline)

Зачастую общая стратегия достижения большей производительности языковых моделей (за исключением некоторых примеров, таких как DistilBERT), сводится к увеличинию количества параметров моделей, а также увеличению обучающей выборки.

Глобально языковые модели из Transformer можно разделить на 3 категории:
* **Автокодирующие модели (Auto-encoding Transformer models)**: BERT-like
    * Особенности:
        * Использует только **энкодер** Трансформера
        * Bi-directional attention, который "видит" все слова в предложении
        * Предобучение таких моделей обычно строится на искажении исходного предложения (например, masked language modeling) и последующей попытке это исходное предложение восстановить
        * Хороши для задач: классификации, NER, extractive question answering
    * Примеры:
        * BERT
        * ALBERT
        * DistilBERT
        * ELECTRA
        * RoBERTa
* **Авторегрессионные модели (Auto-regressive Transformer models)**: GPT-like
    * Особенности:
        * Использует только **декодер** Трансформера
        * Attention декодера не может "заглядывать в будущее": Masked Self-Attention
        * Предобучение таких моделей обычно строится на предсказании следующего слова в предложении
        * Хороши для задач тектовой генерации
    * Примеры:
        * GPT: Generative Pre-trained Transformer
        * CTRL
        * Transformer XL
        * XLNet
* **Модели типа кодировщик-декодировщик (Sequence-to-sequence Transformer models)**: BART/T5-like
    * Особенности:
        * Используются **энкодер и декодер** из Трансформера
        * Стратегия предобучения таких моделей отличается от подхода к подходу
        * Хороши для задач, связанных с созданием новых предложений в зависимости от заданных входных данных: суммаризация, перевод, generative question answering
    * Примеры:
        * BART: bidirectional and auto-regressive transformers
        * mBART
        * Marian
        * T5

## Auto-encoding Transformer models

### BERT: Bidirectional Encoder Representations from Transformers

* [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/pdf/1810.04805.pdf)
* [(Introduction to) Transfer Learning by Lena Voita](https://lena-voita.github.io/nlp_course/transfer_learning.html)
* [What Does BERT Look At? An Analysis of BERT’s Attention](https://arxiv.org/pdf/1906.04341.pdf)

Архитектура BERT (решаем сразу 3 задачи):
* **Transformer's (bidirectional) encoder**
* Next sentence prediction (NSP)
* Masked language modeling (MLM)

#### Bidirectional encoder

Энкодер BERT получает на вход **пары предложений со специальными токенами \[SEP\] и \[CLS\]**:
* \[SEP\] - токен-сепаратор, который ставится в конце предложения A
* \[CLS\] - токен, объединяющий 2 последовательно идущих друг за другом предложения (A и B)

$$\text{[CLS]В дверь постучали 8 раз[SEP]Осьминог, – догадался Штирлиц[SEP]}$$

Входные пары предложений кодируются при помощи эмбеддинга, являющегося суммой 3-х видов входов:
* Token Embedding (WordPiece)
* Position Embedding (позиционное кодирование)
* Segment Embedding (метка 0 для сегмента A или метка 1 для сегмента B)

В энкодере несколько слоев. Последний слой используется для обучения. Также есть вариации, где используется слои из серидины.

#### Next Sentence Prediction 

Это задача бинарной классификации: по токену \[CLS\] модель учится предсказывать, являются ли эти два предложения последовательными предложениями в каком-либо тексте или нет. При этом при обучении 50% примеров содержат последовательные предложения, а другие 50% — случайную пару предложений. 

Пара примеров из оригинальной статьи:
* 1
    * Input: \[CLS\] the man went to \[MASK\] store \[SEP\] he bought a gallon \[MASK\] milk \[SEP\]
    * Label: isNext
* 2
    * Input: \[CLS\] the man went to \[MASK\] store \[SEP\] penguin \[MASK\] are flight ##less birds \[SEP\]
    * Label: notNext

#### Masked language modeling

Алгоритм обучения:
* Выбираем токен из предложения с вероятностью 15%, заменяем их:
    * Специальным токеном \[MASK\] с вероятностью 80%
    * Случайным токеном с вероятностью 10%
    * Оставляем без изменения с вероятностью 10%
* Прогнозируем исходный токен

MLM - это языковая модель, которая видит текст, но некоторые токены "повреждены". При этом это только left-to-right модель, хотя BERT - это Bidirectional encoder.

BERT хорошо понимает кодируемый текст, так как учится заполнять пропуски.

#### Токенизатор WordPiece
 
BERT использует **WordPiece** токенизатор — метод субсловной токенизации, который разбивает редкие слова на более частые подслова.

**Как работает WordPiece:**
- Словарь строится итеративно: начинается с символов, затем добавляются наиболее частые сочетания.
- Редкие слова разбиваются на подслова: например, `"playing"` → `["play", "##ing"]` (символ `##` указывает, что это продолжение слова).
- Это решает проблему OOV (Out-Of-Vocabulary) — модель может обрабатывать слова, которых нет в словаре, разбивая их на известные подслова.

**Пример:**
- `"unpredictable"` → `["un", "##pre", "##dict", "##able"]`
 
#### Варианты BERT

| Параметр | BERT-base | BERT-large |
|----------|-----------|------------|
| **Число слоёв (L)** | 12 | 24 |
| **Скрытая размерность (H)** | 768 | 1024 |
| **Число голов внимания (A)** | 12 | 16 |
| **Общее число параметров** | 110M | 340M |

Для сравнения: оригинальный Transformer имел 6 слоёв энкодера, 512 скрытых нейронов и 8 голов внимания.

#### Входы модели

BERT принимает на вход последовательность слов, которая затем продвигается вверх по стеку энкодеров. Каждый слой энкодера:
1. Применяет Multi-Head Self-Attention.
2. Добавляет Residual Connection и LayerNorm.
3. Применяет Feed-Forward Network.
4. Добавляет Residual Connection и LayerNorm.
5. Передаёт результат следующему слою.

С точки зрения архитектуры, процесс идентичен Transformer encoder. Однако относительно выходов эти две модели значительно отличаются.

#### Выходы модели

Для каждой позиции на выход подаётся вектор размерностью `hidden_size` (768 для BERT-base, 1024 для BERT-large).

**Использование `[CLS]` токена:**
- Для задач классификации предложений (например, определение тональности) используется выход только первой позиции — токена `[CLS]`.
- Этот вектор может быть подан на вход простому классификатору (например, один линейный слой с softmax).
- Авторы BERT достигли SOTA-результатов, используя классификатор с одним слоем.

**Использование выходов всех токенов:**
- Для задач, где важна информация о каждом токене (например, NER, POS-таггинг), используются выходы всех позиций.

#### Модификации

| Модель | Ссылка | Ключевая идея | Основные отличия от BERT | Примечание |
| :--- | :--- | :--- | :--- | :--- |
| **RoBERTa** | [RoBERTa: A Robustly Optimized BERT Pretraining Approach (2019)](https://arxiv.org/abs/1907.11692) | **Robustly optimized BERT** — детальное воспроизведение и оптимизация обучения BERT. | Убрана задача NSP, обучение на большем объёме данных, более длительное обучение с большими батчами, динамическая маскировка. | Показала, что BERT можно значительно улучшить, просто оптимизировав процесс предобучения (больше данных, больше шагов, динамическая маскировка). |
| **DistilBERT** | [DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter (2019)](https://arxiv.org/abs/1910.01108) | **Дистилляция BERT** — сжатая версия, сохраняющая 95% качества. | Размер модели уменьшен на 40% за счёт дистилляции, обучение в 1.5–2 раза быстрее. | Использует технику дистилляции знаний: меньшая модель обучается воспроизводить поведение большой, что позволяет сократить размер модели при минимальной потере качества. |
| **ALBERT** | [ALBERT: A Lite BERT for Self-supervised Learning of Language Representations (2019)](https://arxiv.org/abs/1909.11942) | **A Lite BERT** — лёгкая версия за счёт сокращения числа параметров. | Факторизованная параметризация представления, межслойное разделение параметров, улучшенная задача NSP (SOP). | Добивается сокращения параметров за счёт двух техник: факторизации эмбеддингов и разделения параметров между слоями, что ускоряет обучение в 1.5 раза. Вместо NSP используется задача SOP (Sentence Order Prediction). |
| **SpanBERT** | [SpanBERT: Improving Pre-training by Representing and Predicting Spans (2019)](https://arxiv.org/abs/1907.10529) | **Предобучение на уровне спанов** — модель учится предсказывать целые фрагменты текста. | Вместо отдельных токенов маскируются и предсказываются целые спаны (последовательности). | Особенно полезно для задач, связанных с пониманием связей между фрагментами текста (например, coreference resolution). |
| **ELECTRA** | [ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators (2020)](https://arxiv.org/abs/2003.10555) | **Генеративно-дискриминативное обучение** — более эффективная замена MLM. | Вместо маскирования используется подход «замена токенов»: генератор заменяет токены, дискриминатор определяет, какие из них были заменены. | Предлагает принципиально иной подход: модель учится различать реальные и подставленные токены, что делает обучение более эффективным по сравнению с MLM. |

## Auto-regressive Transformer models

Основное отличие auto-encoding моделей и auto-regressive моделей заключается в направлении обработки контекста: BERT использует **двунаправленный** контекст (видит и прошлое, и будущее), а GPT — **однонаправленный** (видит только прошлое, а будущее кодируется при помощи casual mask). Auto-regressive модели были разработаны для генеративных задач (left-to-right), а также хорошо справляются с few-shot и zero-shot задачами.

Основные недостатки Auto-encoding моделей:
1. **Непригодность для генерации текста**: Auto-encoding модели, как BERT, обучаются восстанавливать исходный текст по его искажённой версии (например, предсказывая замаскированные слова). Это делает их отличными "читателями" для задач понимания языка (NLU), но они не умеют генерировать связный текст с нуля.
2. **Расхождение между предобучением и дообучением (Pre-train/Fine-tune Gap)**: Во время обучения BERT использует специальный токен `[MASK]`, чтобы "скрыть" часть входных данных. Однако при решении реальных задач (fine-tuning) этот токен не используется. Модель сталкивается с данными, которые отличаются от тех, на которых она обучалась, что может снижать её эффективность.
3. **Независимость предсказаний**: При предсказании нескольких замаскированных слов в одном предложении, BERT предсказывает их независимо друг от друга. Это не позволяет модели улавливать сложные корреляции между предсказываемыми токенами, что критично для многих генеративных задач.

### GPT: Generative Pre-trained Transformer

* [(Introduction to) Transfer Learning by Lena Voita](https://lena-voita.github.io/nlp_course/transfer_learning.html)
* [Визуализация архитектур различных GPT-моделей](https://bbycroft.net/llm)
* [GPT in 60 Lines of NumPy](https://jaykmody.com/blog/gpt-from-scratch)

**GPT** (Generative Pre-trained Transformer) — это семейство моделей, представленных OpenAI, начиная с 2018 года в статье [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Архитектура GPT решает следующие ключевые задачи:
- **Generative** — способность генерировать связный текст, а не только классифицировать или понимать его.
- **Pre-trained** — предобучение на больших объёмах неразмеченных текстовых данных.
- **Transformer** — использование архитектуры Transformer.

#### Архитектура: Decoder-only Transformer

GPT использует **только декодер** Transformer. Это делает модель **однонаправленной**: каждый токен может «видеть» только предыдущие токены, но не будущие.

**Ключевые компоненты архитектуры**:

1. **Embedding Layer** — преобразует входные токены в плотные векторы.
2. **Positional Encoding** — добавляет информацию о позиции токенов в последовательности (в оригинальном GPT используется learnable positional encoding).
3. **Transformer Decoder Blocks** — стопка идентичных слоёв (в GPT-1 используется 12 слоёв), каждый из которых содержит:
   - **Masked Multi-Head Self-Attention** — механизм внимания с маской, которая запрещает токенам «подглядывать» в будущее (causal attention).
   - **Feed-Forward Neural Network** — полносвязная сеть для независимой обработки каждого токена.
   - **Layer Normalization** и **Residual Connections** — для стабилизации обучения.
4. **Output Layer** — генерирует вероятности для следующего токена.

**Ключевое отличие от BERT:** GPT использует **Masked Self-Attention** (causal mask), где каждый токен может обращать внимание только на себя и предыдущие токены. Это делает модель авторегрессионной: она предсказывает следующий токен на основе всех предыдущих.

GPT проходит двухэтапное обучение:
1. **Unsupervised Pre-training**
2. **Supervised Fine-tuning**

#### Unsupervised Pre-training

**Цель предобучения:** предсказать следующий токен в последовательности (Causal Language Modeling, CLM).

**Формально:** для последовательности токенов $U = \{u_1, \dots, u_n\}$ модель максимизирует логарифм правдоподобия:

$$L_1(U) = \sum_{i=1}^{n} \log P(u_i \mid u_{i-1}, \dots, u_1; \Theta)$$

#### Supervised Fine-tuning

После предобучения модель донастраивается на конкретные задачи с использованием размеченных данных.

**Функция потерь при донастройке:**

$$L_3 = L_2 + \lambda \cdot L_1$$

где:
- $L_2$ — loss для целевой задачи (например, классификация).
- $L_1$ — loss языкового моделирования (для сохранения способности к генерации).
- $\lambda$ — коэффициент, регулирующий вклад LM-loss.

#### Токенизатор: Byte-Pair Encoding (BPE)

GPT использует **Byte-Pair Encoding (BPE)** для токенизации текста.

**Как работает BPE:**
- Изначально каждый символ (байт) является отдельным токеном.
- Итеративно объединяются наиболее частые пары соседних токенов в один новый токен.
- В результате получается словарь субсловных единиц, который эффективно обрабатывает редкие слова и слова, отсутствующие в словаре.

**Преимущества BPE:**
- Решает проблему OOV (Out-Of-Vocabulary) — редкие слова разбиваются на известные подслова.
- Компактный словарь (в GPT-1 — около 40 000 токенов).
- Эффективно работает с разными языками и доменами.

#### Варианты GPT

| Модель | Год | Число параметров | Архитектура | Ключевое нововведение |
| :--- | :--- | :--- | :--- | :--- |
| **GPT-1** | 2018 | 117M | 12 слоёв, 12 голов внимания | Первая генеративная предобученная модель |
| **GPT-2** | 2019 | 1.5B | 48 слоёв, увеличивающийся масштаб | Zero-shot learning — выполнение задач без донастройки |
| **GPT-3** | 2020 | 175B | 96 слоёв, 96 голов внимания | Few-shot / in-context learning |

GPT принимает на вход последовательность токенов, которая затем продвигается по стеку декодеров. Каждый слой декодера:
1. Применяет **Masked Multi-Head Self-Attention** (с causal маской).
2. Добавляет остаточное соединение и LayerNorm.
3. Применяет **Feed-Forward Network**.
4. Добавляет остаточное соединение и LayerNorm.
5. Передаёт результат следующему слою.

Для каждой позиции на выход подаётся вектор размерностью `hidden_size` (768 для GPT-1, 1600 для GPT-2 Small).

**Использование выходов:**
- Для генерации текста используется выход последнего токена для предсказания следующего.
- Для задач классификации (после донастройки) используется выход последнего токена или специального токена.
- В отличие от BERT, GPT не использует токен `[CLS]` для агрегации информации.

### Другие модели семейства: CTRL, Transformer-XL, XLNet

После успеха GPT исследователи начали искать способы улучшить архитектуру, чтобы устранить её ограничения или расширить возможности. Основные проблемы, которые решали эти вариации:

1. **Отсутствие управляемости генерацией** — GPT генерирует текст на основе начального контекста, но не позволяет явно контролировать стиль, тему или формат вывода.
2. **Ограниченная длина контекста** — стандартный Transformer обрабатывает только фиксированное количество токенов (например, 512), что ограничивает способность модели работать с длинными текстами и удерживать контекст на больших расстояниях.
3. **Ограниченная двунаправленность** — GPT использует только односторонний (left-to-right) контекст, что может быть неоптимально для задач, где важен двусторонний контекст.

#### CTRL: Conditional Transformer Language Model

**CTRL** (Conditional Transformer Language Model) была предложена в 2019 году исследователями из Salesforce Research в статье [CTRL: A Conditional Transformer Language Model for Controllable Generation](https://arxiv.org/abs/1909.05858).

**Ключевая идея:** управляемая генерация текста через **control codes** (управляющие коды).

В отличие от GPT, где генерация определяется только начальным контекстом, CTRL позволяет пользователю **явно задать** желаемый стиль, тему или формат генерируемого текста.

**Как это работает:**
- В начало последовательности добавляется специальный **control code** — токен, определяющий тип генерируемого текста.
- Модель обучается использовать этот код как дополнительное условие для генерации.
- В результате CTRL может генерировать текст в различных стилях: `Reviews` для обзоров, `News` для новостей, `Books` для литературного стиля, `Links` для ссылок и т.д.

**Архитектурные особенности:**
- **Однонаправленный (causal) Transformer** — как и GPT, CTRL является авторегрессионной моделью с causal маской.
- **Control codes** — первый токен в последовательности является управляющим кодом.
- **Размер:** 1.63 миллиарда параметров.
- Обучен на ~140 GB текстовых данных, размеченных с помощью control codes.

**Отличия от GPT:**
- В GPT управление генерацией возможно только через начальный промпт.
- В CTRL управление осуществляется через явные control codes, что даёт более предсказуемый результат.
- CTRL позволяет генерировать текст с заданными характеристиками без необходимости подбирать сложные промпты.

> CTRL полезен в задачах, где требуется контролируемая и предсказуемая генерация: создание структурированных документов, генерация контента в определённом стиле, написание текстов для разных доменов.

#### Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context

**Transformer-XL** была предложена в 2019 году исследователями из Carnegie Mellon University и Google Brain в статье [Transformer-XL: Attentive Language Models Beyond a Fixed-Length Context](https://arxiv.org/abs/1901.02860).

**Ключевая идея:** преодоление ограничения фиксированной длины контекста в стандартных Transformer-моделях.

Стандартный Transformer (включая GPT) обрабатывает текст фрагментами фиксированной длины (например, 512 токенов). Это создаёт две проблемы:
1. Модель не может использовать информацию из предыдущих фрагментов.
2. При переходе между фрагментами теряется связность контекста.

Transformer-XL решает эти проблемы с помощью двух ключевых инноваций.

**Архитектурные особенности:**
1. **Segment-Level Recurrence (рекуррентность на уровне сегментов)** — скрытые состояния (hidden states) предыдущего сегмента сохраняются в памяти и используются при обработке текущего сегмента. Это позволяет модели "помнить" информацию из предыдущих фрагментов текста.
2. **Relative Positional Encoding (относительное позиционное кодирование)** — вместо абсолютных позиций токенов (как в оригинальном Transformer) используется относительное кодирование, которое учитывает расстояние между токенами. Это необходимо, потому что при рекуррентном соединении сегментов абсолютные позиции становятся неоднозначными.

**Отличия от GPT:**
- GPT использует абсолютное позиционное кодирование и обрабатывает текст фрагментами фиксированной длины.
- Transformer-XL использует относительное позиционное кодирование и рекуррентность между сегментами, что позволяет работать с текстом любой длины.
- Transformer-XL обучается на зависимостях, которые на 80% длиннее, чем у RNN, и на 450% длиннее, чем у стандартного Transformer.

**Преимущества:**
- Transformer-XL может обрабатывать последовательности **без ограничения длины**.
- Улучшенное качество на длинных текстах за счёт сохранения контекста между сегментами.
- Более высокая скорость инференса за счёт переиспользования скрытых состояний.

#### XLNet: Generalized Autoregressive Pretraining for Language Understanding

**XLNet** была предложена в 2019 году исследователями из Carnegie Mellon University и Google Brain в статье [XLNet: Generalized Autoregressive Pretraining for Language Understanding](https://arxiv.org/abs/1906.08237).

**Ключевая идея:** объединение преимуществ авторегрессионных (GPT) и двунаправленных (BERT) подходов через **permutation language modeling** (пермутационное языковое моделирование).

**Проблема, которую решает XLNet:**
- **GPT** (авторегрессионная модель) использует только односторонний контекст (left-to-right), что может быть неоптимально для задач, где важен двусторонний контекст.
- **BERT** (автоэнкодинговая модель) использует двунаправленный контекст, но страдает от расхождения между предобучением (с `[MASK]`) и дообучением (без `[MASK]`), а также предсказывает замаскированные токены независимо друг от друга.

XLNet предлагает **обобщённый авторегрессионный подход**, который позволяет модели использовать **двунаправленный контекст**, оставаясь при этом авторегрессионной (и, следовательно, свободной от проблем BERT).

**Архитектурные особенности:**
1. **Permutation Language Modeling (пермутационное языковое моделирование)** — во время обучения модель рассматривает все возможные порядки (перестановки) токенов в последовательности. Это позволяет каждому токену видеть контекст как слева, так и справа, но в рамках авторегрессионной формулировки.
2. **Two-Stream Self-Attention (двухпоточное self-attention)** — специальный механизм внимания, который разделяет информацию о содержимом токена и его позиции, чтобы избежать "подглядывания" в предсказываемый токен.
3. **Использование Transformer-XL** — XLNet заимствует механизм рекуррентности и относительного позиционного кодирования у Transformer-XL, что позволяет ему эффективно работать с длинными последовательностями.
4. **Partial Prediction (частичное предсказание)** — для упрощения обучения предсказываются только `1/K` токенов в каждой перестановке.

**Отличия от GPT:**
- GPT использует только left-to-right порядок токенов.
- XLNet использует все возможные перестановки, что даёт доступ к двунаправленному контексту.
- GPT использует абсолютное позиционное кодирование, XLNet — относительное (как в Transformer-XL).
- XLNet лучше справляется с задачами, где важен двусторонний контекст.

### Недостатки авторегрессионных моделей и KV-cache

Авторегрессионная генерация — это последовательное предсказание каждого следующего токена на основе всех предыдущих. У этого подхода есть ряд фундаментальных ограничений, часть из которых напрямую связана с механизмом KV-cache.

Основные недостатки:
1. **Линейный рост времени генерации**            
   Алгоритмическая сложность генерации линейно зависит от длины генерируемого текста: чтобы сгенерировать \(N\) токенов, нужно выполнить \(N\) последовательных шагов. Каждый шаг требует полного прохода модели, и время генерации увеличивается линейно. Это принципиальное свойство авторегрессии — в отличие от диффузионных моделей, которые могут генерировать несколько токенов параллельно.
2. **Невозможность редактировать начало ответа**            
   Нельзя отредактировать начало ответа, если тема генерации изменилась в конце. Представьте, что вы написали классную концовку к тексту, но понимаете, что под неё лучше изменить начало, — то же самое хотелось бы сделать в LLM. Но авторегрессионная модель не может «вернуться назад»: каждый токен жёстко зафиксирован в момент генерации и не может быть пересмотрен.
3. **Накопление ошибок (exposure bias)**               
   Ошибка, допущенная на раннем шаге, влияет на все последующие. Модель обучается на «правильных» последовательностях (teacher forcing), но на инференсе использует свои собственные предсказания — это создаёт разрыв между обучением и генерацией. Если модель ошиблась, она не может «исправить» ошибку, а лишь продолжает генерировать на основе уже испорченного контекста.
4. **Отсутствие глобального планирования**               
   Модель не может планировать структуру ответа целиком. Она выбирает следующий токен локально, не имея возможности «заглянуть вперёд» и понять, как это повлияет на весь текст. Это приводит к таким проблемам, как зацикливание, потеря логической связности на длинных текстах, трудности с задачами, требующими глобальных ограничений (например, стихосложение с рифмами).
5. **Плохой параллелизм при инференсе**              
   В отличие от обучения, где весь контекст обрабатывается параллельно, генерация строго последовательна. Даже на мощных GPU каждый новый токен требует отдельного прохода модели, что ограничивает пропускную способность.

#### KV-Cache (Key-Value Cache)

**KV-Cache** — это оптимизация, специфичная для **авторегрессивной генерации** (в моделях типа GPT), которая радикально ускоряет инференс.

**Проблема:**
При генерации текста по одному токену, на каждом шаге модель заново вычисляет ключи (K) и значения (V) для **всех** предыдущих токенов, хотя они уже были вычислены на предыдущих шагах. Это приводит к квадратичному росту вычислений $O(n^2)$.

**Решение:**
KV-Cache сохраняет (кэширует) вычисленные ключи и значения для всех предыдущих токенов. При генерации нового токена:
1.  Вычисляются K и V **только для нового токена**.
2.  Они добавляются (конкатенируются) к существующему кэшу.
3.  Внимание вычисляется с использованием полного кэша (старые K/V + новые).

**Результат:**
*   Вычислительная сложность инференса снижается с $O(n^2)$ до **$O(n)$**.
*   Значительно уменьшается latency (задержка) генерации.

**Цена:**
KV-Cache требует **памяти** для хранения K и V всех токенов. Для длинных последовательностей и больших моделей кэш может вырасти до размеров, превышающих веса самой модели, что становится новым узким местом. Это привело к появлению методов **сжатия KV-Cache** (например, эвикшн стратегии, низкоранговые аппроксимации).

> KV-Cache — это **оптимизация для инференса**, которая устраняет избыточные вычисления в авторегрессивных моделях, ускоряя генерацию ценой роста использования памяти.

## Sequence-to-Sequence (Seq2Seq) Transformer модели

**Sequence-to-Sequence (Seq2Seq) Transformer** — это архитектура, которая использует **и энкодер, и декодер** Transformer, объединяя их в единую структуру. Именно такая архитектура была предложена в оригинальной статье **"Attention Is All You Need"** (Vaswani et al., 2017).

В полной архитектуре Transformer энкодер и декодер соединяются последовательно: **сначала весь вход обрабатывается энкодером, затем декодер начинает генерацию**.

`Входная последовательность → Энкодер → (контекстное представление) → Декодер → Выходная последовательность`

#### Энкодер (Encoder)

Энкодер обрабатывает входную последовательность (например, предложение на исходном языке) и преобразуёт её в **контекстное представление** — набор векторов, которые кодируют информацию о каждом токене с учётом всего контекста.

Каждый слой энкодер содержит два подблока:
1. **Multi-Head Self-Attention** — каждый токен может «видеть» все токены входной последовательности (двунаправленное внимание).
2. **Feed-Forward Network** — независимая обработка каждого токена.

В энкодере **нет маски** — все токены видят друг друга, что позволяет создавать богатые контекстные представления.

#### Декодер (Decoder)

Декодер генерирует выходную последовательность пошагово (авторегрессионно). Каждый слой декодера содержит **три** подблока:

1. **Masked Multi-Head Self-Attention** — каждый токен может «видеть» только предыдущие токены (causal mask), что предотвращает «подглядывание» в будущее.
2. **Cross-Attention (Encoder-Decoder Attention)** — запросы (queries) приходят из декодера, а ключи (keys) и значения (values) — из выхода энкодера.
3. **Feed-Forward Network**.

#### Ключевой компонент: Cross-Attention

**Cross-Attention** — это уникальный для Seq2Seq архитектуры механизм, который связывает энкодер и декодер. Он позволяет декодеру на каждом шаге генерации «обращаться» к входной последовательности и выбирать наиболее релевантные части. Без cross-attention декодер был бы просто языковой моделью, не имеющей доступа к исходному тексту.

### Три типа внимания в одном Transformer

| Тип внимания | Queries | Keys / Values | Маска | Назначение |
| :--- | :--- | :--- | :--- | :--- |
| **Encoder Self-Attention** | Из энкодера | Из энкодера | Нет | Построить контекстное представление входа |
| **Decoder Causal Self-Attention** | Из декодера | Из декодера | Causal (будущие токены скрыты) | Авторегрессионная генерация |
| **Cross-Attention** | Из декодера | Из энкодера | Нет (обычно padding mask) | Связать выход с входом |

### Обучение Seq2Seq моделей

Seq2Seq модели обучаются в два этапа:

1. **Pre-training** — модель обучается на больших объёмах неразмеченных текстовых данных с использованием **denoising objectives** (зашумление и восстановление текста). Например, токены могут быть удалены, перемешаны или заменены, а модель учится восстанавливать исходную последовательность.
2. **Fine-tuning** — модель адаптируется к конкретной задаче на размеченных данных.

### Популярные Seq2Seq модели

#### T5 (Text-To-Text Transfer Transformer)

**T5** (Raffel et al., 2020) — модель, которая формулирует **все** NLP задачи как задачи преобразования текста в текст. Одна и та же архитектура используется для перевода, суммаризации, классификации, вопросно-ответных систем и других задач.

**Ключевые особенности:**
- Унифицированный подход: любой вход → текст, любой выход → текст.
- Предобучение на C4 (Colossal Clean Crawled Corpus) — ~750 GB текста.
- Доступны версии разных размеров: T5-Small, T5-Base, T5-Large, T5-3B, T5-11B.

#### BART (Bidirectional and Auto-Regressive Transformers)

**BART** (Lewis et al., 2020) — модель, которая объединяет двунаправленный энкодер (как в BERT) и авторегрессионный декодер (как в GPT).

**Ключевые особенности:**
- Энкодер — двунаправленный (как в BERT).
- Декодер — авторегрессионный (как в GPT).
- Предобучение на **denoising** — текст зашумляется (токены удаляются, перемешиваются, заменяются), и модель учится восстанавливать оригинал.
- Отлично подходит для генеративных задач (суммаризация, перевод, генерация ответов).

### Почему decoder-only модели стали доминирующей архитектурой для LLM

Авторегрессионные модели стали доминирующей архитектурой в современных больших языковых моделях, потеснив encoder-decoder. Это произошло не потому, что encoder-decoder - плохая архитекутра, а потому что decoder-only оказался более удачным компромиссом для задач, где важны масштабируемость, универсальность и эффективность. 

Кроме того, были и другие попытки подойти к задаче генерации при помощи диффузионных моделей и архитектуры вроде Mamba и RWKV.

#### Почему победили decoder-only модели?

Decoder-only архитектура выиграла благодаря ряду системных преимуществ, которые особенно ярко проявились при масштабировании:

1.  **Простота и унификация**: У decoder-only модели **один набор параметров** и нет разделения на энкодер и декодер. Вход (промпт) и выход (ответ) — это **единая последовательность**. Это упрощает и архитектуру, и процесс обучения, и инференс.

2.  **Эффективность обучения**: Каждый токен в последовательности участвует в вычислении функции потерь (предсказание следующего токена). В encoder-only моделях (например, BERT) для обучения используется только около 15% замаскированных токенов, что менее эффективно.

3.  **Предсказуемое масштабирование**: Производительность decoder-only моделей улучшается наиболее предсказуемо при увеличении числа параметров и объема данных.

4.  **Универсальность через промптинг**: Благодаря обучению на предсказание следующего токена, decoder-only модели естественным образом приобрели способность к **In-Context Learning** (обучению по примерам в промпте). Это позволяет решать множество задач (перевод, суммаризация, классификация) без изменения архитектуры или дообучения, просто формулируя задачу как текст.

5.  **Единообразие обучения и инференса**: Модель учится предсказывать следующий токен и на инференсе делает то же самое. Это устраняет разрыв между обучением и использованием, который существует, например, в BERT (где есть токен `[MASK]`, отсутствующий на этапе инференса).

#### Ограничения encoder-decoder, которые стали решающими

У оригинальной encoder-decoder архитектуры есть свои сильные стороны, но для роли универсальной масштабируемой LLM она оказалась менее удобной.

*   **Сложность и ресурсоемкость**: Наличие двух отдельных стеков (энкодера и декодера) делает архитектуру более сложной. При масштабировании это приводит к удвоению числа параметров и вычислительных затрат по сравнению с decoder-only моделью с сопоставимым бюджетом.
*   **Требовательность к данным**: Для эффективного обучения encoder-decoder модели часто нужны **парные данные** (например, «входной текст — целевой текст»). Decoder-only модели могут обучаться на гораздо более доступных неразмеченных текстах.
*   **Специализация, а не универсальность**: Encoder-decoder модели остаются **отличным выбором для конкретных задач** seq2seq, таких как машинный перевод, суммаризация или преобразование структурированных данных. Однако они хуже подходят на роль «универсального солдата», который решает все задачи через промптинг.

#### При чем тут диффузионные модели?

Диффузионные модели - это **принципиально иной класс генеративных моделей**, который возник как альтернатива авторегрессионному подходу, в том числе в текстовой модальности.

Ключевое отличие:
* **Авторегрессия (GPT)** генерирует текст **последовательно, токен за токеном**, слева направо. Каждый новый токен зависит от всех предыдущих. Модель не может «вернуться назад» и исправить ошибку, а генерация плохо параллелится.
* **Диффузия** подходит к генерации как к **итеративному процессу «очистки» от шума**. Модель начинает с полностью замаскированной последовательности и на каждом шаге **параллельно** предсказывает и заполняет все токены, постепенно уточняя результат за несколько проходов.

**Преимущества диффузионных языковых моделей:**
* **Параллельная генерация**: может генерировать несколько токенов за один шаг, снижая задержку.
* **Встроенная коррекция ошибок**: итеративный процесс позволяет исправлять неудачные предсказания.
* **Лучшая управляемость и разнообразие**: диффузионные модели могут генерировать более разнообразные тексты, избегая «зацикливания» на частых фразах.

Диффузионные модели используют **архитектуру Transformer с двунаправленным вниманием** (как в энкодере BERT), что позволяет назвать их «генеративным BERT». Это показывает, что даже в эпоху доминирования авторегрессии архитектурные идеи энкодера не исчезли, а нашли новое применение.

#### Mamba и RWKV: зачем нужны альтернативы?

В контексте доминирования decoder-only Transformer ключевым ограничением остаётся **квадратичная сложность attention** \(O(N^2)\) и **растущий KV-кэш** при инференсе. Именно на решение этих проблем нацелены архитектуры Mamba и RWKV.

**Mamba (State Space Models):**
*   **Идея**: заменить attention на **рекуррентный механизм с непрерывным состоянием** (State Space Model), который обновляется при обработке каждого токена.
*   **Преимущество**: линейная сложность \(O(N)\) и константная память при инференсе.
*   **Ограничение**: из-за сжатия истории в фиксированное состояние модель хуже справляется с точным воспроизведением информации из прошлого (recall) и симметричными паттернами.

**RWKV (Linear Attention):**
*   **Идея**: объединить **эффективность обучения Transformer** с **эффективностью инференса RNN** через линейный механизм внимания.
*   **Преимущество**: константная память при инференсе (модель работает как RNN) и параллелизуемое обучение (как Transformer).
*   **Ограничение**: переносит в продолжение только ограниченную информацию из промпта, что снижает качество на задачах с длинным контекстом.

**Как они вписываются в ландшафт:**
Mamba и RWKV — это не замена Transformer, а **альтернативы для конкретных сценариев**: очень длинные последовательности, edge-устройства, потоковая обработка. Из-за своих ограничений они чаще используются в **гибридных архитектурах**, где комбинируются с attention (например, Jamba использует соотношение 1:7 attention к Mamba).

#### Итого

Ландшафт архитектур последовательностей становится всё более разнообразным:
* **Decoder-only Transformer** остаётся доминирующей архитектурой благодаря универсальности, In-Context Learning и предсказуемому масштабированию.
* **Encoder-decoder** нашли нишу в специализированных задачах (перевод, суммаризация), но проиграли в универсальности.
* **Диффузионные модели** предлагают принципиально иной подход с параллельной генерацией и коррекцией ошибок, используя при этом двунаправленный Transformer.
* **Mamba и RWKV** решают проблему квадратичной сложности attention, но ценой ограничений в recall, что делает их нишевыми решениями и основой для гибридных архитектур.

Наиболее перспективным направлением, по-видимому, являются **гибридные архитектуры**, комбинирующие сильные стороны разных подходов.